In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- Block Influence, properly calibrated. ~15 min total, no training.
#
# The BI numbers we have been reading came from a 120-molecule THPep smoke test.
# This redoes them on 1,200 molecules drawn from AmpHGT, PAMPA and THPep, because
# the teacher's behaviour varies several-fold across these distributions and the
# ranking inside the low-influence trough is what we are about to act on.
#
# BI_i = 1 - E_{x,t}[ cos( block input, block output ) ] over non-pad tokens.
#
# Scored on TWO models:
#   the full 32-block teacher     -- the reference ranking
#   prefix16 (blocks 0-15)        -- the model we would actually prune, since
#                                    prefix16 is the best THPep arm so far (0.8531,
#                                    above the full 32-block model's 0.7764)
# BI is a property of the residual stream, and truncating changes that stream, so
# a ranking taken on the 32-block model is not automatically valid for the
# 16-block one. Measuring both is cheap and settles it.
subprocess.run('pip install -q -U "transformers>=5.0"', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
assert torch.cuda.device_count() >= 1, "need a GPU"


In [ ]:

# -- Cell 3 -- code, data, teacher.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

# probe_bi.py resolves data and models from a project root; mirror that layout.
if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)

# THPep_train/test must exist for probe_bi's THPep calibration source.
subprocess.run(["python", "bench_control.py"], cwd=CODE, capture_output=True, text=True)
assert os.path.exists(CODE + "/probe_bi.py")
print("ok -- layout mirrored")


In [ ]:

# -- Cell 4 -- export prefix16, the second scoring target.
EXPORT = WORK + "/compressed"
PREFIX16 = EXPORT + "/peptideclm-2-mlm-prefix16"
os.makedirs(EXPORT, exist_ok=True)
if not os.path.exists(PREFIX16 + "/model.safetensors"):
    r = subprocess.run(["python", "export_truncated.py", "--out", PREFIX16,
                        "--keep", "0-15"], cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-600:])
    if r.returncode != 0:
        print(r.stderr[-1200:])
    assert r.returncode == 0, "prefix16 export failed"
print("prefix16 ready")


In [ ]:

# -- Cell 5 -- score BI on both models. ~5 min each.
RES = WORK + "/results"
os.makedirs(RES, exist_ok=True)
CALIB = "AmpHGT,PAMPA,THPep"
N_PER = 400

targets = [("full32", TEACH), ("prefix16", PREFIX16)]
for tag, path in targets:
    out = "%s/block_influence_%s.json" % (RES, tag)
    if os.path.exists(out):
        print("%s: already scored" % tag); continue
    t0 = time.time()
    r = subprocess.run(["python", "-u", "probe_bi.py", "--model", path,
                        "--calib", CALIB, "--n-per", str(N_PER),
                        "--batch", "16", "--out", out],
                       cwd=CODE, capture_output=True, text=True)
    print("===== %s (%.1f min) =====" % (tag, (time.time() - t0) / 60))
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print("----- STDERR -----"); print(r.stderr[-2000:])
    assert r.returncode == 0, "BI failed for " + tag


In [ ]:

# -- Cell 6 -- per-block residual norms, for the seam-mismatch risk column.
#
# WHY THIS MATTERS MORE THAN BI. The slice probe showed the thing that actually
# breaks a pruned model is the distribution jump at the seam: suffix16 (blocks
# 16-31 fed raw embeddings) saw a 39.6x norm mismatch and scored 0.5593, below a
# bag-of-tokens baseline. Interior drops are in a different regime entirely --
# around 1.1-1.5x -- but the exact ratio depends on which blocks go, so measure
# the norms rather than assume.
#
# Dropping blocks a..b means block b+1 receives the stream as it was after block
# a-1 instead of after block b. The ratio of those two norms is the jump.
NORM_PY = CODE + "/_norms.py"
open(NORM_PY, "w").write(r"""
import sys, glob, numpy as np, pandas as pd, torch
sys.path.insert(0, "/kaggle/working/distill")
from transformers import AutoModel, AutoTokenizer
from student import patch_sdpa_dtype
from probe_bi import load_calibration

patch_sdpa_dtype()
src = sys.argv[1]
tok = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
m = AutoModel.from_pretrained(src, trust_remote_code=True,
                              use_safetensors=True).cuda().eval()
core = m.model
blocks = core.transformer.blocks
smis = load_calibration(["AmpHGT", "PAMPA", "THPep"], 100)

st = {"m": None, "den": None, "acc": []}
def hook(mod, inp, out):
    st["acc"].append((((out.float() * st["m"]).sum(1) / st["den"])
                      .norm(dim=1).mean().item()))
hs = [b.register_forward_hook(hook) for b in blocks]

tot = np.zeros(len(blocks) + 1); nb = 0
with torch.no_grad():
    for i in range(0, len(smis), 16):
        e = tok(smis[i:i+16], return_tensors="pt", padding=True,
                truncation=True, max_length=512)
        ids = e["input_ids"].cuda(); att = e["attention_mask"].cuda()
        st["m"] = (ids != 0).unsqueeze(-1).float()
        st["den"] = st["m"].sum(1).clamp(min=1.0)
        st["acc"].clear()
        emb = core.embed(ids).float()
        e0 = ((emb * st["m"]).sum(1) / st["den"]).norm(dim=1).mean().item()
        with torch.autocast("cuda", dtype=torch.float16):
            m(input_ids=ids, attention_mask=att)
        tot += np.array([e0] + st["acc"]); nb += 1
for h in hs:
    h.remove()
np.save(sys.argv[2], tot / nb)
print("NORMS " + " ".join("%.1f" % x for x in tot / nb))
""")

for tag, path in targets:
    npy = "%s/norms_%s.npy" % (RES, tag)
    if os.path.exists(npy):
        continue
    r = subprocess.run(["python", "-u", NORM_PY, path, npy],
                       cwd=CODE, capture_output=True, text=True)
    print("%s: %s" % (tag, r.stdout.strip().splitlines()[-1][:300] if r.stdout else ""))
    if r.returncode != 0:
        print(r.stderr[-1500:])


In [ ]:

# -- Cell 7 -- the table to choose from. BLOCKS 1-16 ONLY.
#
# Block 0 is excluded: its BI is ~195x the median of the middle, and the slice
# probe showed every slice that does not start at block 0 falls below a
# bag-of-tokens baseline. Blocks 17+ are excluded at your request.
LO, HI = 1, 16

for tag, _ in targets:
    f = "%s/block_influence_%s.json" % (RES, tag)
    if not os.path.exists(f):
        continue
    j = json.load(open(f))
    bi = np.array(j["bi"])
    hi = min(HI, len(bi) - 1)
    n = np.load("%s/norms_%s.npy" % (RES, tag)) if os.path.exists("%s/norms_%s.npy" % (RES, tag)) else None

    sel = list(range(LO, hi + 1))
    rank = {b: r + 1 for r, b in enumerate(sorted(sel, key=lambda i: bi[i]))}
    print("\n" + "=" * 68)
    print("%s -- %d blocks, calibrated on %s (%d molecules)"
          % (tag, len(bi), j["calib"], j["n_molecules"]))
    print("=" * 68)
    print("%-6s %10s %6s   %s" % ("block", "BI", "rank", "relative (within 1-%d)" % hi))
    mx = bi[sel].max()
    for b in sel:
        print("%-6d %10.5f %6d   %s" % (b, bi[b], rank[b], "#" * int(round(45 * bi[b] / mx))))
    print("   [block 0 = %.5f, excluded]" % bi[0])
    print("   spread within 1-%d: %.1fx  (%.5f .. %.5f)"
          % (hi, bi[sel].max() / bi[sel].min(), bi[sel].min(), bi[sel].max()))

    order = sorted(sel, key=lambda i: bi[i])
    print("\n   lowest-BI blocks in removal order: %s" % order)
    if n is not None:
        print("\n   contiguous candidates (seam = norm expected / norm received):")
        print("   %-12s %6s %10s %8s   %s" % ("drop", "left", "params M", "seam", "risk"))
        per = 10.49
        for a in range(LO, hi + 1):
            for b in (a + 3, a + 7, a + 11):
                if b > hi:
                    continue
                if a not in order[:len(order)//2 + 2]:
                    continue
                left = len(bi) - (b - a + 1)
                ratio = n[b + 1] / n[a]
                risk = "low" if ratio < 1.5 else ("moderate" if ratio < 2.2 else "HIGH")
                print("   %-12s %6d %10.1f %7.2fx   %s"
                      % ("%d-%d" % (a, b), left, 0.83 + per * left, ratio, risk))

subprocess.run("rclone copy %s %s/results/bi_probe --drive-chunk-size 64M -P"
               % (RES, REMOTE), shell=True, check=False)
print("\nuploaded -> %s/results/bi_probe" % REMOTE)
print("\nPick drop sets from the table and I will wire them into a benchmark run.")
